In [ ]:
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

In [ ]:
import kagglehub
import glob

In [ ]:
path = kagglehub.dataset_download("ikramshah512/amazon-products-sales-dataset-42k-items-2025")

Using Colab cache for faster access to the 'amazon-products-sales-dataset-42k-items-2025' dataset.


In [ ]:
csv_file = glob.glob(f"{path}/*.csv")[0]

In [ ]:
df = pd.read_csv(csv_file)
df.head()

,product_title,product_rating,total_reviews,purchased_last_month,discounted_price,original_price,is_best_seller,is_sponsored,has_coupon,buy_box_availability,delivery_date,sustainability_tags,product_image_url,product_page_url,data_collected_at,product_category,discount_percentage
0,BOYA BOYALINK 2 Wireless Lavalier Microphone f...,4.6,375.0,300.0,89.68,159.00,No Badge,Sponsored,Save 15% with coupon,Add to cart,2025-09-01,Carbon impact,https://m.media-amazon.com/images/I/71pAqiVEs3...,https://www.amazon.com/sspa/click?ie=UTF8&spc=...,2025-08-21 11:14:29,Phones,43.60
1,"LISEN USB C to Lightning Cable, 240W 4 in 1 Ch...",4.3,2457.0,6000.0,9.99,15.99,No Badge,Sponsored,No Coupon,Add to cart,2025-08-29,NaN,https://m.media-amazon.com/images/I/61nbF6aVIP...,https://www.amazon.com/sspa/click?ie=UTF8&spc=...,2025-08-21 11:14:29,Laptops,37.52
2,"DJI Mic 2 (2 TX + 1 RX + Charging Case), Wirel...",4.6,3044.0,2000.0,314.00,349.00,No Badge,Sponsored,No Coupon,Add to cart,2025-09-01,NaN,https://m.media-amazon.com/images/I/61h78MEXoj...,https://www.amazon.com/sspa/click?ie=UTF8&spc=...,2025-08-21 11:14:29,Laptops,10.03
3,"Apple AirPods Pro 2 Wireless Earbuds, Active N...",4.6,35882.0,10000.0,162.24,162.24,Best Seller,Organic,No Coupon,NaN,NaN,NaN,https://m.media-amazon.com/images/I/61SUj2aKoE...,https://www.amazon.com/Apple-Cancellation-Tran...,2025-08-21 11:14:29,Phones,0.00
4,Apple AirTag 4 Pack. Keep Track of and find Yo...,4.8,28988.0,10000.0,72.74,72.74,No Badge,Organic,No Coupon,NaN,NaN,NaN,https://m.media-amazon.com/images/I/61bMNCeAUA...,https://www.amazon.com/Apple-MX542LL-A-AirTag-...,2025-08-21 11:14:29,Phones,0.00


In [ ]:
df.columns

Index(['product_title', 'product_rating', 'total_reviews',
       'purchased_last_month', 'discounted_price', 'original_price',
       'is_best_seller', 'is_sponsored', 'has_coupon', 'buy_box_availability',
       'delivery_date', 'sustainability_tags', 'product_image_url',
       'product_page_url', 'data_collected_at', 'product_category',
       'discount_percentage'],
      dtype='object')

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42675 entries, 0 to 42674
Data columns (total 17 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   product_title         42675 non-null  object 
 1   product_rating        41651 non-null  float64
 2   total_reviews         41651 non-null  float64
 3   purchased_last_month  32164 non-null  float64
 4   discounted_price      40613 non-null  float64
 5   original_price        40613 non-null  float64
 6   is_best_seller        42675 non-null  object 
 7   is_sponsored          42675 non-null  object 
 8   has_coupon            42675 non-null  object 
 9   buy_box_availability  28022 non-null  object 
 10  delivery_date         30692 non-null  object 
 11  sustainability_tags   3408 non-null   object 
 12  product_image_url     42675 non-null  object 
 13  product_page_url      40606 non-null  object 
 14  data_collected_at     42675 non-null  object 
 15  product_category   

In [ ]:
feature_cols = df[['product_rating', 'total_reviews', 'discounted_price', 'original_price', 'discount_percentage']].copy()

In [ ]:
product_cat = df[['product_category']].copy()

In [ ]:
y = df['purchased_last_month'].copy()

In [ ]:
y.fillna(0, inplace=True)
feature_cols.fillna(0, inplace=True)
product_cat.fillna('Unknown', inplace= True)

In [ ]:
X = pd.concat([feature_cols, product_cat], axis=1)
X.columns

Index(['product_rating', 'total_reviews', 'discounted_price', 'original_price',
       'discount_percentage', 'product_category'],
      dtype='object')

In [ ]:
X_encoded = pd.get_dummies(X, columns=['product_category'], drop_first=True)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)

In [ ]:
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

RandomForestRegressor(random_state=42)

In [ ]:
predictions = model.predict(X_test)

In [ ]:
print(f"R2: {r2_score(y_test, predictions)}")
print(f"MAE: {mean_absolute_error(y_test, predictions)}")

R2: 0.8942253852867981
MAE: 248.7389302987935


In [ ]:
def predict_sales(product_rating, total_reviews, discounted_price, original_price, discount_percentage, product_category):
  input_df = pd.DataFrame(0, index=[0], columns=X_encoded.columns)

  input_df['product_rating'] = product_rating
  input_df['total_reviews'] = total_reviews
  input_df['discounted_price'] = discounted_price
  input_df['original_price'] = original_price
  input_df['discount_percentage'] = discount_percentage

  formatted_cat = product_category.title()
  category = f"product_category_{formatted_cat}"

  if category in input_df.columns:
    input_df.loc[0, category] = 1

  prediction = model.predict(input_df)

  return f"Estimated Product Sales in a Month: {int(np.round(prediction[0]))} units"

In [ ]:
predict_sales(4, 160, 25, 50, 50, 'Electronics')

'Estimated Product Sales in a Month: 227 units'